In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, year, month

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive"

In [ ]:
spark = SparkSession.builder \
    .appName("Amazon Reviews EDA") \
    .config("spark.sql.files.maxPartitionBytes", "64m") \
    .config("spark.sql.shuffle.partitions", "128") \
    .getOrCreate()

In [ ]:
reviews_path = "/content/drive/MyDrive/Amazon Project - Draft 01/outputs/reviews_eda_sample.parquet"
df_reviews = spark.read.parquet(reviews_path)
items_path = "/content/drive/MyDrive/Amazon Project - Draft 01/outputs/meta.parquet"
df_items = spark.read.parquet(items_path)

In [ ]:
df_reviews.head()
print("Reviews columns:")
print(df_reviews.columns)

In [ ]:
df_items.head()
print("Items columns:")
print(df_items.columns)

In [ ]:
df_reviews = (
    df_reviews
    .withColumnRenamed("parent_asin", "item_id")
    .withColumnRenamed("images", "review_images")
)

df_items = df_items.withColumnRenamed("parent_asin", "item_id")

In [ ]:
df = df_reviews.join(df_items, on="item_id", how="left")

In [ ]:
df.head()

In [ ]:
# Step 1: Extract review text from Spark DataFrame into Pandas
pdf = df.select("text").limit(1000).toPandas()

# Step 2: Clean text and tokenize
import re
from collections import Counter

words = []
for text in pdf["text"].dropna():
    clean = re.sub(r"[^\w\s]", "", text.lower())  # Remove punctuation and convert to lowercase
    words.extend(clean.split())

# Step 3: Load stopwords from sklearn and add custom stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop_words = set(ENGLISH_STOP_WORDS)
custom_stopwords = {
    "use", "just", "like", "dont", "little", "br", "very", "its",
    "you", "have", "not", "are", "bought", "product","really","using","need","doe"
}
stop_words.update(custom_stopwords)

# Step 4: Lemmatize words to unify different forms (e.g., "using" → "use")
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in words]

# Step 5: Remove stopwords and empty strings
filtered = [w for w in lemmatized if w not in stop_words and w.strip() != ""]

# Step 6: Count word frequency
word_counts = Counter(filtered)
top_words = word_counts.most_common(20)

# Step 7: Visualize top keywords using horizontal bar chart with vertical gridlines
import matplotlib.pyplot as plt

labels, values = zip(*top_words)
plt.figure(figsize=(10,6))
plt.barh(labels[::-1], values[::-1], color="skyblue")
plt.title("Top 20 Keywords in Reviews (Lemmatized)")
plt.xlabel("Frequency")
plt.grid(axis='x', linestyle='--', alpha=0.5)  # Add vertical gridlines
plt.tight_layout()
plt.show()

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import size

df_img_count = df.withColumn("num_images", size("review_images"))

df_img_count.groupBy("num_images").count().orderBy("num_images").show()

In [ ]:
from pyspark.sql.functions import size, avg
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: add images number column
df_with_count = df.withColumn("num_images", size("review_images"))

# Step 2: group by item_id and compute the average images and average rating
agg_df = df_with_count.groupBy("item_id").agg(
    avg("num_images").alias("avg_images"),
    avg("rating").alias("avg_rating")
)

# Step 3: transfer to Pandas DataFrame
pdf = agg_df.toPandas()

In [ ]:
from pyspark.sql.functions import size, avg
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df_with_count = df.withColumn("num_images", size("review_images"))

agg_df = df_with_count.groupBy("item_id").agg(
    avg("num_images").alias("avg_images"),
    avg("rating").alias("avg_rating")
)

pdf = agg_df.toPandas()

# Step 4: set avg_images bins
bins = [0, 0.5, 1, 2, 3, 5, 10]
labels = ["0", "0.5", "1–2", "2–3", "3–5", "5+"]
pdf["image_bin"] = pd.cut(pdf["avg_images"], bins=bins, labels=labels, include_lowest=True)

# Step 5: Visualization
plt.figure(figsize=(10,6))
sns.violinplot(x="image_bin", y="avg_rating", data=pdf, palette="Blues")
plt.title("Rating Distribution by Avg Review Image Count (Binned)")
plt.xlabel("Average Number of Images per Review (Binned)")
plt.ylabel("Average Rating")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline

# add sentiment analysis model
sentiment_model = pipeline("sentiment-analysis")

In [ ]:
# Step 3: Define Spark UDF to apply sentiment model
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_sentiment(text):
    if text:
        try:
            return sentiment_model(text)[0]['label']
        except:
            return None
    return None

sentiment_udf = udf(get_sentiment, StringType())

# Step 4: Apply UDF to your Spark DataFrame
# Assuming your DataFrame is named df and the review column is 'text'
df_with_sentiment = df.withColumn("sentiment", sentiment_udf(df["text"]))

In [ ]:
pdf = df_with_sentiment.select("text", "rating", "timestamp", "sentiment").toPandas()

In [ ]:
pdf["sentiment"].value_counts().plot(kind="bar", color="skyblue")
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="sentiment", y="rating", data=pdf, palette="Set2")
plt.title("Rating Distribution by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Rating")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
pdf["date"] = pd.to_datetime(pdf["timestamp"], unit="ms")  # adjust if needed
sentiment_trend = pdf.groupby([pdf["date"].dt.to_period("M"), "sentiment"]).size().unstack().fillna(0)

sentiment_trend.plot(marker="o", figsize=(10,6))
plt.title("Sentiment Trend Over Time")
plt.xlabel("Month")
plt.ylabel("Review Count")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
user_category_rating = df.groupBy("user_id", "main_category").agg(
    avg("rating").alias("avg_rating"),
    count("*").alias("review_count")
)